# 🤖 BRACU ML Final Project: AI-Powered Knowledge Chatbot (RAG)
### Strict Domain Grounding, Ingestion Pipeline & Out-of-Scope Fallbacks

This Google Colab notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline designed according to the BRACU Machine Learning lecture materials:
1. **Custom Knowledge Base**: Ingests multi-format documents (PDFs, Markdown notes, text).
2. **Strict Grounding**: Answers questions strictly using indexed materials and outputs granular citations (e.g. `Source: Account_Security.pdf`).
3. **Graceful Out-of-Scope Fallback**: Detects out-of-domain questions using cosine similarity thresholds ($	au$) and declines politely without hallucinations.
4. **Interactive Chat UI**: Renders an interactive chat widget directly in the notebook output matching the mobile app interface.


In [ ]:
# Step 1: Install Dependencies
!pip install -q sentence-transformers scikit-learn pypdf gradio
print('Dependencies installed successfully!')


## 🧠 1. Theoretical Foundations: Attention & Vector Search
From the lecture slides:
- **Scaled Dot-Product Attention**:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
- **Cosine Similarity for Semantic Retrieval**:
$$\text{sim}(q, k) = \frac{q \cdot k}{\|q\| \|k\|}$$
When vectors are $L_2$-normalized, cosine similarity directly equals the vector dot product!


In [ ]:
# Step 2: Define Sample Knowledge Base Documents
KNOWLEDGE_BASE = {
    "Account_Security.pdf": """
According to the "Account Security" document, you can reset your password by clicking the "Forgot Password" link on the login page and following the instructions sent to your registered email.

Two-Factor Authentication (2FA) is strongly recommended for all accounts. When enabled, users must provide a 6-digit one-time verification passcode in addition to their password during login.

After 5 consecutive failed login attempts, your account will be temporarily locked for 15 minutes to protect against brute-force intrusion.
""",
    "01_Sequence_to_Sequence_and_Attention.md": """
Sequence-to-Sequence (Seq2Seq) models (Sutskever et al., 2014) map an input sequence to an output sequence using an Encoder and Decoder.
In traditional Seq2Seq, the encoder compresses the entire input sentence into a single final hidden state vector, creating an Information Bottleneck that causes translation quality (BLEU score) to degrade severely on long sentences (> 20-30 words).

The Attention Mechanism (Bahdanau et al., 2015) solves the bottleneck by allowing the decoder to dynamically attend to all encoder hidden states at each generation step. It computes query-key alignment scores e_{t,i}, normalizes them into attention weights alpha_{t,i} using softmax, and computes a dynamic context vector c_t as a weighted sum of encoder states.

Attention score functions include Additive/MLP attention (Bahdanau), Bilinear (Luong), Dot Product (Luong), and Scaled Dot-Product (Vaswani et al.).
""",
    "02_Transformers_and_BERT.md": """
Transformers process all tokens simultaneously in parallel. Because recurrence is eliminated, Positional Encoding is required to inject sequential token order.
Sinusoidal positional encodings use sine and cosine functions across frequencies: PE(pos, 2i) = sin(pos / 10000^(2i/d)) and PE(pos, 2i+1) = cos(pos / 10000^(2i/d)).

Self-Attention projects input embeddings into Query (Q), Key (K), and Value (V) matrices. The output is computed as Attention(Q,K,V) = softmax(Q K^T / sqrt(d_k)) V.
Multi-Head Attention runs multiple projections in parallel to capture distinct linguistic relationships (e.g. coreference, syntax, causality).
Masked Attention applies a causal triangular mask to prevent future token leakage during autoregressive decoding.

BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only model pre-trained on Masked Language Modeling (MLM, 15% masked) and Next Sentence Prediction (NSP).
""",
    "03_Deep_Generative_Modeling.md": """
Deep Generative Modeling aims to learn a probability distribution p_data(x) to estimate densities and generate novel samples.
Traditional Autoencoders (AEs) compress data into a deterministic bottleneck z, but lack a continuous regularized manifold for random generation.

Variational Autoencoders (VAEs) formulate a probabilistic latent space where the encoder outputs mean mu and variance sigma of a Gaussian distribution. The loss function optimizes Reconstruction Loss plus KL Divergence against a standard normal prior N(0, I).

The Reparameterization Trick solves the non-differentiable sampling problem by expressing z = mu + sigma * epsilon, where epsilon ~ N(0, I). This isolates the randomness into an external node, allowing standard gradient backpropagation through the encoder.

Generative Adversarial Networks (GANs) train a Generator G and Discriminator D in a minimax game: min_G max_D V(D, G).
"""
}

print(f'Defined {len(KNOWLEDGE_BASE)} knowledge documents.')


In [ ]:
# Step 3: Text Chunking and Vector Indexing
import re
import numpy as np
from sentence_transformers import SentenceTransformer

def chunk_document(text, doc_name, chunk_size=350, overlap=50):
    paragraphs = [p.strip() for p in text.strip().split('\n\n') if p.strip()]
    chunks = []
    for idx, p in enumerate(paragraphs):
        chunks.append({
            'doc_name': doc_name,
            'chunk_id': idx,
            'content': p
        })
    return chunks

# Chunk all documents
all_chunks = []
for doc_name, doc_text in KNOWLEDGE_BASE.items():
    all_chunks.extend(chunk_document(doc_text, doc_name))

print(f'Total chunks created: {len(all_chunks)}')

# Load embedding model and compute normalized vector embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [c['content'] for c in all_chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)

print(f'Computed {chunk_embeddings.shape[0]} embeddings of dimension {chunk_embeddings.shape[1]}.')


In [ ]:
# Step 4: Strict RAG Grounding Engine with Threshold Fallback
SIMILARITY_THRESHOLD = 0.32

def answer_query(query: str):
    # 1. Encode query
    query_vec = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    
    # 2. Compute Cosine Similarity (Dot product of normalized vectors)
    scores = np.dot(chunk_embeddings, query_vec)
    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    top_chunk = all_chunks[best_idx]
    
    # 3. Scope Evaluation
    in_scope = (best_score >= SIMILARITY_THRESHOLD)
    
    if in_scope:
        answer = top_chunk['content']
        source = top_chunk['doc_name']
        status = 'GROUNDED'
    else:
        answer = (
            f"I apologize, but this information is not found in the provided knowledge base (Confidence: {best_score:.1%}).\n\n"
            "I am programmed to only answer questions based on the verified documents.\n"
            "Available topics include: Account Security, Seq2Seq & Attention, Transformers & BERT, and VAEs & GANs."
        )
        source = "None (Out of Scope)"
        status = 'OUT_OF_SCOPE'
        
    return {
        'query': query,
        'answer': answer,
        'source': source,
        'confidence': best_score,
        'status': status
    }

# Test with the exact prompt from Image 1!
test_res = answer_query('How do I reset my password?')
print('--- TEST QUERY RESULT ---')
print('Answer:', test_res['answer'])
print('Source Citation:', test_res['source'])
print('Confidence:', test_res['confidence'])


In [ ]:
# Step 5: Interactive Web UI inside Google Colab (Matching Image 1)
import gradio as gr

def gradio_chat(user_message, history):
    res = answer_query(user_message)
    
    if res['status'] == 'GROUNDED':
        bot_response = f"{res['answer']}\n\n---\n`Source: {res['source']}` (Confidence: {res['confidence']*100:.1f}%)"
    else:
        bot_response = res['answer']
        
    return bot_response

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🤖 KnowledgeBot - AI Assistant",
    description="AI-powered chatbot with strict knowledge grounding. Trained on Account Security and BRACU ML lecture topics.",
    examples=[
        ["How do I reset my password?"],
        ["Explain the reparameterization trick in Variational Autoencoders."],
        ["What is the information bottleneck problem in Seq2Seq?"],
        ["What are the pretraining tasks in BERT?"],
        ["What is the recipe for chicken pasta?"]
    ]
)

# Launch interactive interface inside Colab
demo.launch(share=True, debug=False)


In [ ]:
# Step 6: Evaluation & Verification Benchmark
test_suite = [
    ("How do I reset my password?", "GROUNDED", "Account_Security.pdf"),
    ("Explain the reparameterization trick in VAE", "GROUNDED", "03_Deep_Generative_Modeling.md"),
    ("What is Teacher Forcing in Seq2Seq?", "GROUNDED", "01_Sequence_to_Sequence_and_Attention.md"),
    ("What are the two pretraining objectives of BERT?", "GROUNDED", "02_Transformers_and_BERT.md"),
    ("Who won the 2022 football world cup?", "OUT_OF_SCOPE", "None (Out of Scope)"),
    ("How to make chocolate pancakes at home?", "OUT_OF_SCOPE", "None (Out of Scope)"),
]

print(f"{'Query':<50} | {'Expected':<12} | {'Predicted':<12} | {'Score':<8} | {'Status'}")
print("-" * 95)
correct = 0

for q, expected_status, expected_src in test_suite:
    res = answer_query(q)
    is_correct = (res['status'] == expected_status)
    if is_correct:
        correct += 1
    status_icon = "✅ PASS" if is_correct else "❌ FAIL"
    print(f"{q[:48]:<50} | {expected_status:<12} | {res['status']:<12} | {res['confidence']:<8.4f} | {status_icon}")

accuracy = (correct / len(test_suite)) * 100
print("-" * 95)
print(f"Overall Accuracy: {accuracy:.1f}% ({correct}/{len(test_suite)} tests passed)")
